# F1 · ETL Capa 2 — Detalles por partido (stats por jugador/mapa)Ejecuta la **Capa 2** (lenta, tier *expensive* 20/min) con *checkpoint/resume*.Descarga `/v2/match/{id}/details` para cada partido de `data/interim/matches` yaplana a tres tablas: `match_details`, `match_maps` y `match_player_stats`.Aquí se obtienen los **team_id** autoritativos (`teams[].id`).

## 1. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Rutas

In [ ]:
import sys
from pathlib import Path

# Ajusta esta ruta a donde tengas el repo dentro de tu Drive.
DRIVE_BASE = "/content/drive/MyDrive/Analisis de datos/Proyecto final"
# Si prefieres el disco local de Colab (más rápido, efímero), descomenta:
# DRIVE_BASE = "/content/Proyecto final"

REPO_DIR = Path(DRIVE_BASE)
DATA_DIR = REPO_DIR / "data"
SRC_DIR = REPO_DIR / "src"
API_DIR = REPO_DIR / "vlrggapi"
BASE_URL = "http://127.0.0.1:3001"

for d in (REPO_DIR, DATA_DIR, SRC_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC_DIR))
print("REPO_DIR:", REPO_DIR)
print("DATA_DIR:", DATA_DIR)


## 3. Dependencias + clonar vlrggapi

In [ ]:
# Dependencias del proyecto (pandas, pyarrow, requests, tqdm, ...).
!pip install -q -r "{REPO_DIR}/requirements.txt"
# Dependencias de vlrggapi (uv no está en Colab; usamos pip directamente).
!pip install -q fastapi "uvicorn[standard]" httpx selectolax cachetools

# Clona vlrggapi (la instancia hosteada está caída). Si ya existe, no pasa nada.
!git clone https://github.com/axsddlr/vlrggapi.git "{API_DIR}" 2>/dev/null || echo "vlrggapi ya clonado"


## 4. Servidor + cliente

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

from scraper import VLRClient, start_server, wait_for_api

# Arranca el servidor si no responde (idempotente entre notebooks/runtime).
if not wait_for_api(BASE_URL, timeout=5):
    proc = start_server(API_DIR, port=3001, log_file=DATA_DIR / "_vlrggapi.log")
    wait_for_api(BASE_URL, timeout=240)

client = VLRClient(base_url=BASE_URL, data_dir=DATA_DIR)
print("API lista:", client.version())


## 5. Ejecutar Capa 2

In [ ]:
from etl import run_layer2

# Para una prueba rápida con los primeros N partidos, descomenta:
# tables = run_layer2(client, limit=500)
# Sin limit: procesa TODOS los match_id de data/interim/matches.
tables = run_layer2(client)


## 6. Sanity check

In [ ]:
for name, df in tables.items():
    print(f"{name:20s} {df.shape}")

print("\nColumnas de match_details:")
print(list(tables["match_details"].columns))
print("\nColumnas de match_player_stats:")
print(list(tables["match_player_stats"].columns))
print("\nEjemplo player_stats:")
print(tables["match_player_stats"].head())
